In [4]:
from __future__ import annotations

import re

In [ ]:
# 1. Imports and functions
import pandas as pd

from pathlib import Path
import sys

print("Notebook working folder:", Path.cwd())

module_folder = Path.cwd().parent
print("Module found:", (module_folder / "validation_structure.py").exists())

sys.path.insert(0, str(module_folder))

from validation_structure import normaliser, valider

# 2. Load the full dataset
df = pd.read_csv("../data/numeros_tva.csv", dtype={"numero_tva": "string"})

# 3. Normaliser and valider toutes les lignes
df["numero_tva_normalise"] = df["numero_tva"].fillna("").map(normaliser)
df["validation"] = df["numero_tva_normalise"].map(valider)
df["numero_tva_valide"] = df["validation"].map(lambda resultat: resultat[0])

Notebook working folder: c:\Users\Utilisateur\Documents\tva_intracommunautaire\formation-data-IA\99-Brief\Data-Analyst\TVA-Intracommunautaire\starter-kit\notebook
Module found: True


In [22]:
import pandas as pd

# valider les numéros de TVA normalisés et stocker le résultat dans une nouvelle colonne

df = pd.read_csv("../data/numeros_tva.csv")
df["numero_tva_normalise"] = df["numero_tva"].fillna("").apply(normaliser)

In [24]:
# initialiser le dataframe avec les données de TVA

df = pd.read_csv("../data/numeros_tva.csv")
print("le nombre total des lignes chargées:", len(df))

le nombre total des lignes chargées: 10000


In [ ]:
# Nettoyage, normalisation et validation des numéros de TVA

placeholders = {"", "NONE", "NULL", "NA", "NAN", "NOTAPPLICABLE"}

df["numero_tva_normalise"] = df["numero_tva"].fillna("").apply(normaliser)
df["validation"] = df["numero_tva_normalise"].apply(valider)
df["numero_tva_valide"] = df["validation"].map(lambda resultat: resultat[0])

valides = df.loc[
    ~df["numero_tva_normalise"].isin(placeholders)
    & df["numero_tva_valide"]
].copy()


In [ ]:
# verifier les doublons parmi les numéros valides

counts = valides["numero_tva_normalise"].value_counts()
doublons = counts[counts > 1]

print(doublons)

numero_tva_normalise
DK71704289        6
DK91970813        4
NL785427788B39    4
PT100405711       4
PL2944016503      4
                 ..
FI91136826        2
DK28569033        2
NL082237803B06    2
DK89608228        2
PL3605374689      2
Name: count, Length: 253, dtype: int64


In [12]:
# verifier les doublons des lignes parmi les doublouns de DK71704289

df.loc[
    df["numero_tva_normalise"].eq("DK71704289"),
    ["numero_tva_normalise", "raison_sociale", "pays_declare", "date_saisie", "source_saisie"]
]

,numero_tva_normalise,raison_sociale,pays_declare,date_saisie,source_saisie
219,DK71704289,Lumen Technologies Sp. z o.o.,DK,2025-03-12,reprise_erp
300,DK71704289,Lumen Technologies Sp. z o.o.,DK,2024-07-18,reprise_erp
495,DK71704289,Lumen Technologies Sp. z o.o.,DK,2024-05-15,reprise_erp
1417,DK71704289,Lumen Technologies Sp. z o.o.,DK,2025-01-17,reprise_erp
2412,DK71704289,Lumen Technologies Sp. z o.o.,DK,2024-02-09,portail_client
5361,DK71704289,Lumen Technologies Sp. z o.o.,DK,2024-10-05,reprise_erp


In [29]:
# verifier les doublons exacts parmi les lignes valides

exact_duplicates = df[df.duplicated(keep=False)]
print(exact_duplicates)

Empty DataFrame
Columns: [id, raison_sociale, pays_declare, numero_tva, date_saisie, source_saisie, numero_tva_normalise, validation, numero_tva_valide, pays_code, chiffre_tva]
Index: []


In [31]:
# élimination des doublons des lignes qui contiennent 
# des numeros de tva identiques

valides = valides.loc[
    ~valides["numero_tva_normalise"].duplicated(keep=False)
].copy()

In [33]:

pays_exclu = {"GB", "ZZ", "UK", "XX", "QQ"}

pays = df["pays_declare"].astype("string").str.strip().str.upper()
exclu = pays.isin(pays_exclu)

print("lignes supprimées:", exclu.sum())
df = df.loc[~exclu].copy()

lignes supprimées: 519


In [45]:
print("lignes exclues:", exclu.sum())
print("lignes retenues:", (~exclu).sum())
print("lignes sans prefix parmi les lignes retenues:", without_prefix.sum())

lignes exclues: 519
lignes retenues: 9481
lignes sans prefix parmi les lignes retenues: 532


In [54]:
df_reference = df.loc[~exclu].copy()

numero_tva = df_reference["numero_tva_normalise"].fillna("").str.strip().str.upper()
pays = df_reference["pays_declare"].astype("string").str.strip().str.upper()

sans_prefixe = vat.ne("") & ~vat.str.match(r"^[A-Z]{2}", na=False)

# These country codes use the same prefix as their declared country code.
prefix_map = {
    "NL": "NL", "SE": "SE", "PL": "PL", "PT": "PT", "DK": "DK",
    "BE": "BE", "FI": "FI", "FR": "FR", "IT": "IT", "LU": "LU",
}
prefixe = country.map(prefix_map)
to_prefix = missing_prefix & prefix.notna()

df_reference.loc[to_prefix, "numero_tva_normalise"] = (
    prefix[to_prefix] + vat[to_prefix]
)

print("Lignes dans le jeu de données de référence :", len(df_reference))
print("Préfixes ajoutés :", to_prefix.sum())
print("Préfixes manquants restants :", (sans_prefixe & prefixe.isna()).sum())

Lignes dans le jeu de données de référence : 9481
Préfixes ajoutés : 532
Préfixes manquants restants : 0


In [ ]:
# séparation du numero_tva en pays_code et chiffre_tva

numero_tva = (
    df_reference["numero_tva_normalise"]
    .fillna("")
    .str.strip()
    .str.upper()
)

df_reference["pays_code"] = numero_tva.str[:2]
df_reference["chiffre_tva"] = numero_tva.str[2:]


In [ ]:
# 1. production d'un échantillon au hasard de 20 numero_tva

sample = valides.sample(n=min(20, len(valides)))

# 2. séparation du 'numero_tva_normalise' en 'pays_code' et 'chiffre_tva'
sample["pays_code"] = sample["numero_tva_normalise"].str[:2]
sample["chiffre_tva"] = sample["numero_tva_normalise"].str[2:]

print(sample[["numero_tva_normalise", "pays_code", "chiffre_tva"]])


     numero_tva_normalise pays_code   chiffre_tva
596            FI75234493        FI      75234493
6550         BE1643041032        BE    1643041032
37             FI26606369        FI      26606369
7154        IT30309490354        IT   30309490354
8584           LU95568402        LU      95568402
241         IT96780897581        IT   96780897581
1255       SE703635039001        SE  703635039001
6739         PL8111258763        PL    8111258763
3026          PT130441090        PT     130441090
5317          PT559329938        PT     559329938
285         FR54171718047        FR   54171718047
4176           LU99427960        LU      99427960
9735         PL9961783002        PL    9961783002
6651       SE626483611201        SE  626483611201
3218           DK71641309        DK      71641309
7346        IT89972942778        IT   89972942778
6997           FI28235610        FI      28235610
4780        FR65725972591        FR   65725972591
6647        FR82800274047        FR   82800274047


In [62]:
import pandas as pd

# 1. le nombre total des lignes dans le jeu de données nettoyés
print(f"le total des lignes dans 'valides': {len(valides)}\n")

# 2. nombre et pourcentage par pays ('pays_code')
liste_pays = pd.DataFrame({
    "count": valides["pays_code"].value_counts(),
    "percentage (%)": (valides["pays_code"].value_counts(normalize=True) * 100).round(2)
})

print(liste_pays.to_string())


le total des lignes dans 'valides': 5549

           count  percentage (%)
pays_code                       
LU           586           10.56
PL           585           10.54
PT           580           10.45
BE           572           10.31
FR           565           10.18
SE           557           10.04
DK           534            9.62
NL           533            9.61
IT           526            9.48
FI           511            9.21
